# Fundamentals 05 - Lineage Memory API

Este notebook **explora la API** de Lineage Memory. No es un caso OTC; es una pieza core para convertir un `RunResult` en memoria compacta y explicable.

Lineage Memory responde tres preguntas:

```text
qué pasó -> cómo pasó -> por qué la respuesta está soportada
```

También sirve para pasar contexto compacto a una siguiente llamada sin reenviar todo el trace bruto.

In [ ]:
from __future__ import annotations

import agentic_systems as lab

PRETTY = False

## Problema default de fundamentals

Todos los notebooks de `tutorials/` usan este mismo problema para que puedas comparar la API sin cambiar de caso cada vez:

```text
Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final
```



In [ ]:
USER_PROMPT = """Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2.
Dime:
- procedimiento
- resultado final""".strip()

# La estructura solicitada por el usuario se materializa como datos simples.
# No es un parser ni una respuesta precocinada: sólo representa la sección `Dime:`.
REQUESTED_OUTPUTS = ["procedimiento", "resultado_final"]

lab.show({
    "prompt_usuario": USER_PROMPT,
    "salidas_solicitadas": REQUESTED_OUTPUTS,
}, title="Problema default · visible")


## 1) Crear un resultado mínimo

Usamos una tool simple con `python-direct` porque `fundamentals` explora la API sin depender de credenciales, Athena ni Bedrock.

In [ ]:
@lab.tool
def add_numbers(a: float, b: float) -> dict:
    """Suma dos números y devuelve evidencia estructurada."""
    value = a + b
    return {
        "ok": True,
        "operation": "add",
        "a": a,
        "b": b,
        "result": value,
        "summary": f"{a} + {b} = {value}",
    }


result = add_numbers.run({"a": 10, "b": 32})
result.final = lab.final_answer(result.data, schema=lab.output_schema(["operation", "result"]))

lab.show({"run_result": result.normalized()})

## 2) Construir `LineageMemory`

`LineageMemory` no ejecuta nada. Sólo proyecta el `RunResult` a una memoria compacta, explicable y serializable.

In [ ]:
memory = lab.LineageMemory.from_run_result(
    result,
    name="fundamentals.add_numbers",
    question="Problema aritmético default",
    goal="Explicar qué tool se ejecutó, qué evidencia produjo y por qué la respuesta es válida.",
    tags=["fundamentals", "lineage"],
)

lab.show(memory, title="Lineage Memory · explicación")

lab.show(memory.compact(), title="Lineage Memory · payload serializable")

## 3) Explicación: qué, cómo y por qué

`explain()` prepara una vista directa para notebooks, revisiones humanas y reportes de ejecución.

In [ ]:
explanation = memory.explain()
lab.show(explanation)

## 4) Contexto compacto para ahorrar tokens

`to_prompt_context(...)` es la forma corta de llevar memoria a una llamada posterior. En vez de pasar todo `trace(full)`, pasas sólo los hechos importantes.

In [ ]:
compact_context = memory.to_prompt_context(max_chars=900)
savings = memory.estimated_context_savings(result.trace("full"), max_chars=900)

print(compact_context)
lab.show(savings)

## 5) Método directo desde `RunResult`

Para uso diario, `result.lineage(...)` es el atajo recomendado.

In [ ]:
same_memory = result.lineage(
    name="fundamentals.add_numbers.shortcut",
    question="Problema aritmético default",
    goal="Mostrar el atajo desde RunResult.",
)

lab.show(same_memory, title="Lineage Memory · atajo desde RunResult")

lab.show({
    "schema_version": same_memory.schema_version,
    "steps": [step.kind for step in same_memory.steps],
    "answer": same_memory.answer,
}, title="Lineage Memory · campos mínimos")

## 6) Convivencia con contratos/policies

Lineage Memory complementa `ContractPolicySpec`: el contrato dice qué debía pasar; la memoria explica qué pasó y qué evidencia lo soporta.

In [ ]:
spec = lab.ContractPolicySpec(
    name="fundamentals.add_numbers.contract",
    contract=lab.AgentContract(
        must_call=["add_numbers"],
        tool_expectation=lab.expect.exactly("add_numbers"),
        expected_tool_outputs={"add_numbers": {"ok": True, "operation": "add"}},
    ),
    policy=lab.RunPolicy(max_turns=2, max_tool_calls=1, trace="compact"),
)

result.validation = result.validate(spec.contract).to_dict()
contract_memory = result.lineage(
    name="fundamentals.add_numbers.with_contract",
    question="Problema aritmético default",
    goal="Mostrar contrato + lineage juntos.",
)

lab.show(spec.describe(), title="ContractPolicySpec")
lab.show(contract_memory, title="Lineage Memory · contrato + ejecución")

## 7) Render humano

`human_result` sigue renderizando la ejecución. Con `show_lineage=True`, además muestra la sección **Qué pasó · Lineage Memory** sin cambiar el contrato base de la API. `LineageMemory` sigue siendo la memoria explicable que puedes guardar, resumir o pasar al siguiente prompt.

In [ ]:
lab.human_result(
    result,
    title="Fundamentals · RunResult con Lineage Memory",
    expected_tools=spec.contract.tool_expectation,
    pretty=PRETTY,
    show_lineage=True,
    lineage=contract_memory,
)

lab.show({"prompt_context": contract_memory.to_prompt_context(max_chars=900)}, title="Contexto compacto para siguiente prompt")

## Lineage del problema default

Mismo caso, pero empaquetado como una sola tool local para enfocar el notebook en `LineageMemory`: explicación, evidencia y contexto compacto.

In [ ]:
@lab.tool
def solve_default_problem() -> dict:
    """Resuelve el problema aritmético default con procedimiento."""
    return user_problem_payload()


default_result = solve_default_problem.run({})
default_result.final = lab.final_answer(
    default_result.data,
    schema=lab.output_schema(fields=["procedimiento", "resultado_final", "ok"]),
)

default_lineage = default_result.lineage(
    name="fundamentals.default_problem.lineage",
    question=USER_PROMPT,
    goal="Mostrar Lineage Memory sobre el problema default común a todos los notebooks.",
)

lab.human_result(
    default_result,
    title="Human result + Lineage Memory · problema default",
    expected_tools=lab.expect.exactly("solve_default_problem"),
    pretty=PRETTY,
    show_lineage=True,
    lineage=default_lineage,
)

lab.show(default_lineage.estimated_context_savings(default_result.trace("full"), max_chars=900))


## Lo importante

- `fundamentals` explora la API; no mete dominio OTC.
- `LineageMemory` se construye desde `RunResult`.
- La memoria explica `qué`, `cómo` y `por qué`.
- Sirve para reportar resultados y para reducir contexto en llamadas posteriores.
- No reemplaza observabilidad ni tracing externo; es una proyección compacta y portable.

## Coverage API de este notebook

Esta tabla deja explícito qué parte de Agentic Systems queda materializada aquí.

In [ ]:
api_coverage = [
    {
        "api": "LineageMemory.from_run_result",
        "description": "Convierte un resultado en huella auditable de ejecucion."
    },
    {
        "api": "result.lineage",
        "description": "Atajo directo para derivar lineage desde el resultado."
    },
    {
        "api": "explain",
        "description": "Explica el flujo con lenguaje humano y trazable."
    },
    {
        "api": "human_text",
        "description": "Genera un texto humano breve y estable desde la huella."
    },
    {
        "api": "to_prompt_context",
        "description": "Empaqueta contexto compacto para ahorrar tokens."
    },
    {
        "api": "estimated_context_savings",
        "description": "Mide cuanto contexto conserva o recorta el lineage."
    },
    {
        "api": "default problem lineage",
        "description": "Aplica lineage al mismo problema base del tutorial."
    }
]

lab.show({'notebook': '05_lineage_memory_api.ipynb', 'api_coverage': api_coverage})